# RAGbattre Project Showcase
This notebook demonstrates how to use the main modules and scripts of the RAGbattre project for importing, processing, analyzing, and querying parliamentary debate data. It covers the full backend pipeline, including ChromaDB, LLM backends, and utility functions.

## 1. Environment Setup & Imports
Import all necessary libraries and project modules for data processing, retrieval, and LLM interaction.

In [1]:
import os
import pandas as pd
import numpy as np
import chromadb
from chromadb.utils import embedding_functions

from retrieval_app.core import (
    initialize_chromadb,
    query_documents,
    get_available_collections,
    load_example_questions,
    query_seance,
    query_documents_filtered,
    query_documents_regex_filtering,
    query_documents_reranking,
    extract_document_data,
    BASE_DIR, DATA_DIR, DEFAULT_QUERY, DEFAULT_COLLECTION, DEFAULT_EMBEDDING_MODEL, EMBEDDINGS_DIR, EXAMPLE_QUESTIONS_FILE, CORPUS_DIR
)
from retrieval_app.llm_utils import (
    get_available_models,
    get_ollama_response,
    get_ollama_response_mistral,
    get_llm_response
)

/home/wac/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/home/wac/anaconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/wac/anaconda3/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/home/wac/anaconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/wac/anaconda3/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on 

## 2. Data Import and Corpus Exploration
Explore the available corpus and example questions. This is the starting point for any analysis or retrieval.

In [2]:
# List available sessions (collections) and inspect the corpus
corpus_dir = CORPUS_DIR
print("Available corpus files:", os.listdir(corpus_dir)[:5])

# Load example questions
questions = load_example_questions(EXAMPLE_QUESTIONS_FILE)
print("Sample question:", questions[0])

Available corpus files: ['1881-01-20.txt']
Sample question: {'question': 'Selon M. le président, quand la séance a-t-elle été ouverte ?', 'source': 'La séance est ouverte à deux heures un quart.', 'file_name': '1881-01-11', 'bloc_source': 'PRÉSIDBNCE DE M. DESSEAUX, DOYEN D\'AGE La séance est ouverte à deux heures un quart.\n\nM. le président. Aux termes de l\'article 1er de la loi constitutionnelle du 16 juillet 1875, je déclare ouverte la session ordinaire de la Chambre des députés pour 1881.\n\nJ\'invite lts six membres les plus jeunes de \'\'Assemblée à vouloir bien répondre à \'l\'appel de leur nom pour prendre place au bureau en qualité de secrétaires provisoires.\n\n(L\'appel des noms des députés les plus jeunes est fait par un huissier.)\n\nSont successivement appelés : MM. Georges de Cassagnac, né le 17 févrièr 1855; Adrien Bastii, né Je 1er octobre 1853; Jules André, né le 23 août 1852 ; René Gautier, né le \'25 avril 1852 ; Emile Réaux, né le 20 juin 1851 ; Le Provost de Lau

## 3. Chunking and Preprocessing (Script Usage)
Show how to split raw documents into chunks using the provided script. This is required before generating embeddings.

In [3]:
# Example: Run the chunking script (usually done once)
# !python ../scripts/split_corpus_cs1.py

# After running, check the output directory
splitted_dir = os.path.join(DATA_DIR, "corpus_splitted_cs1")
print("Sample chunked session:", os.listdir(splitted_dir)[:1])
print("Chunks in session:", os.listdir(os.path.join(splitted_dir, os.listdir(splitted_dir)[0]))[:3])

Sample chunked session: ['1881-01-20']
Chunks in session: ['1881-01-20_008.txt', '1881-01-20_006.txt', '1881-01-20_005.txt']


## 4. Embedding Generation (Script Usage)
Generate embeddings for the chunked documents using the provided script. This populates the ChromaDB vector database.

In [4]:
# Example: Run the embedding generation script (usually done once)
# !python ../scripts/embeddings/generate_embeddings_cs1.py

# After running, check the embeddings directory
print("ChromaDB embeddings directory:", os.listdir(EMBEDDINGS_DIR))

ChromaDB embeddings directory: ['chroma.sqlite3', 'a0eedeb0-5f45-4588-b474-a6b086820597']


## 5. ChromaDB: Connecting and Querying
Initialize ChromaDB, list available collections, and perform a basic semantic search.

In [ ]:
client = chromadb.PersistentClient(path=EMBEDDINGS_DIR)
collections = get_available_collections(client)
print("Available collections:", collections)

# Initialize a collection for querying
collection_name = collections[0]
client, collection, embedding_fn = initialize_chromadb(collection_name, DEFAULT_EMBEDDING_MODEL, client=client)

# Query documents (semantic search)
query = "Qui est le président de la séance ?"
docs, ids = query_documents(query, collection, n_results=3)
print("Top retrieved docs:", docs)

Available collections: ['1881-01-20']


## 6. Advanced Retrieval: Filtering, Regex, and Reranking
Demonstrate advanced retrieval options: keyword filtering, regex, and reranking.

In [ ]:
# Keyword filtering
filtered_docs, filtered_ids = query_documents_filtered(query, collection, word_to_filter="président", n_results=3)
print("Filtered docs:", filtered_docs)

# Regex filtering
regex_docs, regex_ids = query_documents_regex_filtering(query, collection, regex_pattern="président.*séance", n_results=3)
print("Regex-filtered docs:", regex_docs)

# Reranking
reranked_indices, reranked_docs = query_documents_reranking(query, collection, n_results=3)
print("Reranked docs:", reranked_docs)

## 7. LLM Backends: Ollama, Mistral, Cohere
Show how to use the unified LLM interface to generate answers from different backends.

In [ ]:
# Prepare messages for LLMs
messages = [
    {"role": "user", "content": "Qui est le président de la séance ?"}
]

# Ollama (local)
ollama_models = get_available_models()
print("Available Ollama models:", ollama_models)
response_ollama = get_ollama_response(model=ollama_models[0], messages=messages)
print("Ollama response:", response_ollama)

# Mistral (cloud)
response_mistral = get_ollama_response_mistral(messages)
print("Mistral response:", response_mistral)

# Unified interface (auto-selects backend)
response_auto = get_llm_response(model=ollama_models[0], messages=messages)
print("Unified LLM response:", response_auto)

## 8. RAG Pipeline: Retrieval-Augmented Generation
Combine retrieval and generation for a full RAG workflow, as in the Streamlit app.

In [ ]:
# Retrieve context documents
context_docs, context_ids = query_documents(query, collection, n_results=3)

# Build LLM prompt with context
system_prompt = "Tu es un assistant expert en débats parlementaires."
rag_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Contexte: {context_docs}\n\nQuestion: {query}"}
]

# Generate answer using the preferred backend
rag_response = get_llm_response(model=ollama_models[0], messages=rag_messages)
print("RAG response:", rag_response)

## 9. Utility Functions and Data Extraction
Demonstrate utility functions such as extracting structured data from LLM output and loading example questions.

In [ ]:
# Extract structured data from a generated response (if applicable)
sample_output = '{"document_id": "1881-01-20_003", "text": "Le président de la séance est M. Dupont."}'
parsed = extract_document_data(sample_output)
print("Parsed document data:", parsed)

# Load and display example questions
example_questions = load_example_questions(EXAMPLE_QUESTIONS_FILE)
print("Example questions:", example_questions[:2])

## 10. References & Further Reading
- See the project README and CLAUDE.md for more details on architecture, configuration, and advanced usage.
- For more advanced experiments, see `notebooks/rag.ipynb` and `multi_hop/multihop_tests.ipynb`.